# iSWAP gate with an explicit SQUID-loop tunable coupler

Two fluxonium qubits are coupled through a SQUID-loop transmon whose Josephson
energy is flux-tunable.  Rather than treating the coupler as a classical control
line, it is included as a full quantum degree of freedom in the Hilbert space.
The coupler Josephson energy follows

$$E_J^{(c)}(\Phi_c) \;=\; E_{J,\max}^{(c)} \, |\cos(\pi\,\Phi_c/\Phi_0)|,$$

and it is connected capacitively to both qubits,

$$H_{\rm int} \;=\; g_A\, n_A n_c \;+\; g_B\, n_B n_c.$$

When the coupler is far detuned and remains in its ground state, it mediates an
effective $n_A n_B$ coupling whose strength depends on $\Phi_c$.  Modulating the
coupler flux at the difference frequency
$\omega_d = \omega_{gf}^{(A)} - \omega_{gf}^{(B)}$,

$$\Phi_c(t) = \Phi_{\rm dc} + \Phi_{\rm ac}\cos(\omega_d t),$$

therefore parametrically drives the $|gf\rangle \leftrightarrow |fg\rangle$
exchange -- provided the qubits are biased slightly off the integer-flux symmetry
point so that $\langle g|n|f\rangle \neq 0$.

The main entry point is `simulate_gate(phi_ac, phi_dc, T_gate)`, whose three
control parameters are:

| symbol | meaning | units |
|---|---|---|
| `phi_ac` | peak AC coupler-flux modulation amplitude | $\Phi_0$ |
| `phi_dc` | DC coupler-flux bias | $\Phi_0$ |
| `T_gate` | gate duration | ns |

It returns the Z-corrected gate fidelity $F$, the error $1-F$, peak leakage
into coupler-excited and $|ee\rangle$ states, the derived $J_{\rm mod}^{\rm eff}$
at the operating point, and population traces throughout the gate.

In [ ]:
import sys, os, time
sys.path.insert(0, os.getcwd())

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

import qchard_fluxonium as fluxonium
import qchard_evolgates as gates

## 1. Physical parameters

Both qubits use a heavy-fluxonium parameter set: $E_J = 4.97$ GHz,
$E_C = 1.5$ GHz, $E_L = 1.0$ GHz.  Qubit A sits at integer flux
($\phi_{\rm ext} = 0$); qubit B has $E_J$ shifted slightly upward so that
its $g$-$f$ transition is detuned by approximately $100$ MHz from qubit A.
Both qubits are additionally offset from the symmetry point by a static phase
`phi_offset_qubit` to render $\langle g|n|f\rangle$ nonzero -- a prerequisite
for the parametric coupling to have any matrix element in the $g$-$f$ subspace.

The SQUID-loop transmon coupler is parameterised with $E_{J,\max} = 22$ GHz and
$E_C^{(c)} = 0.25$ GHz, placing its bare frequency around $6$ GHz at zero flux
and tunable downward toward $\Phi_0/2$.  Capacitive qubit-coupler couplings of
$g_A = g_B = 80$ MHz are used throughout.  Hilbert space sizes are kept small
(`nlev_q = 4`, `nlev_c = 4`) -- sufficient to capture the dominant leakage
channels while keeping the propagator calculation tractable.

In [ ]:
E_J_A, E_C_A, E_L_A = 4.97, 1.5, 1.0
E_J_B, E_C_B, E_L_B = 4.97 + 0.06, 1.5, 1.0

phi_offset_qubit = 0.5

nlev_q  = 4
nlev_lc = 30

qA = fluxonium.Fluxonium(E_L=E_L_A, E_C=E_C_A, E_J=E_J_A,
                         phi_ext=phi_offset_qubit,
                         nlev=nlev_q, nlev_lc=nlev_lc)
qB = fluxonium.Fluxonium(E_L=E_L_B, E_C=E_C_B, E_J=E_J_B,
                         phi_ext=phi_offset_qubit,
                         nlev=nlev_q, nlev_lc=nlev_lc)

E_J_max_c = 22.0
E_C_c     = 0.25
nlev_c    = 4
ncut_c    = 30
asym_d    = 0.0

g_Ac = 0.080
g_Bc = 0.080

for name, q in (("A", qA), ("B", qB)):
    eg, ee, ef = q.freq(0,1), q.freq(0,2), q.freq(0,2)
    print(f"qubit {name}:  w_ge = {q.freq(0,1)*1e3:7.2f} MHz, "
          f"w_gf = {q.freq(0,2)*1e3:7.2f} MHz, "
          f"<g|n|f> = {abs(q.n_ij(0,2)):.4f}")

qubit A:  w_ge = 6935.18 MHz, w_gf = 11347.96 MHz, <g|n|f> = 0.0753
qubit B:  w_ge = 6974.82 MHz, w_gf = 11402.67 MHz, <g|n|f> = 0.0751


## 2. SQUID-loop transmon coupler

The coupler is constructed in the charge basis with cutoff $n_{\rm cut}$
(basis states $|n = -n_{\rm cut}\rangle, \dots, |n = +n_{\rm cut}\rangle$):

$$ H_c(\Phi_c) \;=\; 4 E_C^{(c)} \, \hat n_c^{\,2}
   \;-\; E_J^{(c)}(\Phi_c)\, \cos\hat\varphi_c.$$

For a symmetric SQUID, $E_J^{(c)}(\Phi_c) = E_{J,\max} \cos(\pi\Phi_c/\Phi_0)$.
The sign of this expression is immaterial for the spectrum; it matters only for
the sign of off-diagonal matrix elements.

The operator that the AC flux modulation couples to at leading order is the
flux derivative of the Hamiltonian,

$$\frac{\partial H_c}{\partial \Phi_c}
   \;=\; E_{J,\max}\,\pi\,\sin(\pi\Phi_c/\Phi_0)\,\cos\hat\varphi_c.$$

This is computed analytically in the charge basis and then projected into the
truncated coupler eigenbasis at the chosen $\Phi_{\rm dc}$.

In [ ]:
class SquidTransmon:
    Flux-tunable transmon coupler (symmetric SQUID, charge basis).
    def __init__(self, E_J_max, E_C, nlev=4, ncut=30, asym=0.0):
        self.E_J_max = E_J_max
        self.E_C = E_C
        self.nlev = nlev
        self.ncut = ncut
        self.asym = asym
        dim = 2*ncut + 1
        self._n_charge = qt.Qobj(np.diag(np.arange(-ncut, ncut+1, dtype=float)))
        shift = np.zeros((dim, dim))
        for k in range(dim-1):
            shift[k+1, k] = 1.0
        shift = qt.Qobj(shift)
        self._cos_phi_charge = 0.5 * (shift + shift.dag())
        self._sin_phi_charge = -0.5j * (shift - shift.dag())

    def _E_J(self, phi_c):
        if self.asym == 0.0:
            return self.E_J_max * np.cos(np.pi * phi_c)
        d = self.asym
        c = np.cos(np.pi * phi_c)
        s = np.sin(np.pi * phi_c)
        return self.E_J_max * np.sqrt(c*c + (d*s)**2) * np.sign(c)

    def H_charge(self, phi_c):
        return 4 * self.E_C * self._n_charge**2 - self._E_J(phi_c) * self._cos_phi_charge

    def dHdPhi_charge(self, phi_c):
        return self.E_J_max * np.pi * np.sin(np.pi * phi_c) * self._cos_phi_charge

    def diag(self, phi_c):
        Return (evals[nlev], evecs[dim, nlev]) of the bare coupler.
        H = self.H_charge(phi_c)
        ev, es = H.eigenstates()
        return ev[:self.nlev], es[:self.nlev]

    def project(self, op_charge, phi_c):
        Project a charge-basis operator into the truncated eigenbasis.
        _, es = self.diag(phi_c)
        n = self.nlev
        M = np.zeros((n, n), dtype=complex)
        for i in range(n):
            for j in range(n):
                M[i, j] = (es[i].dag() * op_charge * es[j])[0, 0]
        return qt.Qobj(M)

    def n_op(self, phi_c):
        return self.project(self._n_charge, phi_c)

    def H_diag(self, phi_c):
        ev, _ = self.diag(phi_c)
        ev = ev - ev[0]
        return qt.Qobj(np.diag(ev))

    def dHdPhi_diag(self, phi_c):
        return self.project(self.dHdPhi_charge(phi_c), phi_c)


coupler = SquidTransmon(E_J_max=E_J_max_c, E_C=E_C_c,
                        nlev=nlev_c, ncut=ncut_c, asym=asym_d)

phi_scan = np.linspace(0, 0.49, 50)
freq_01_c = np.zeros_like(phi_scan)
freq_12_c = np.zeros_like(phi_scan)
for k, p in enumerate(phi_scan):
    ev, _ = coupler.diag(p)
    freq_01_c[k] = ev[1] - ev[0]
    freq_12_c[k] = ev[2] - ev[1]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(phi_scan, freq_01_c, label=r"$\omega_{01}^{(c)}$")
ax.plot(phi_scan, freq_12_c, label=r"$\omega_{12}^{(c)}$")
ax.set_xlabel(r"$\Phi_{dc}/\Phi_0$")
ax.set_ylabel("Frequency (GHz)")
ax.set_title("SQUID-transmon coupler spectrum")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Building the full three-system Hilbert space

The total Hilbert space $\mathcal H_A \otimes \mathcal H_B \otimes \mathcal H_c$
has dimension `nlev_q` $\times$ `nlev_q` $\times$ `nlev_c` $= 64$.

All operators are assembled at the static coupler flux $\Phi_{\rm dc}$, with the
AC modulation treated separately as a time-dependent perturbation.  Three
objects are retained:

- `H_static` -- the bare qubit and coupler Hamiltonians plus the capacitive
  qubit-coupler interaction $g_A n_A n_c + g_B n_B n_c$.
- `dH_mod` -- $I_A \otimes I_B \otimes (\partial H_c / \partial \Phi_c)$
  evaluated at $\Phi_{\rm dc}$; this is the operator the flux modulation drives.
- Charge operators `nA_full`, `nB_full`, `nc_full` for diagnostics.

In [ ]:
def build_system(phi_dc):
    Assemble static H, modulation operator dH/dPhi, and diagnostics.
    nA = qA.n()
    nB = qB.n()
    H_A = qt.Qobj(np.diag(qA.levels()[:nlev_q] - qA.levels()[0]))
    H_B = qt.Qobj(np.diag(qB.levels()[:nlev_q] - qB.levels()[0]))

    H_c   = coupler.H_diag(phi_dc)
    n_c   = coupler.n_op(phi_dc)
    dH_c  = coupler.dHdPhi_diag(phi_dc)

    Ia = qt.qeye(nlev_q); Ib = qt.qeye(nlev_q); Ic = qt.qeye(nlev_c)

    H_A_f = qt.tensor(H_A, Ib,   Ic)
    H_B_f = qt.tensor(Ia,   H_B, Ic)
    H_c_f = qt.tensor(Ia,   Ib,  H_c)
    nA_f  = qt.tensor(nA,   Ib,  Ic)
    nB_f  = qt.tensor(Ia,   nB,  Ic)
    nc_f  = qt.tensor(Ia,   Ib,  n_c)
    dH_f  = qt.tensor(Ia,   Ib,  dH_c)

    H_int    = g_Ac * nA_f * nc_f + g_Bc * nB_f * nc_f
    H_static = H_A_f + H_B_f + H_c_f + H_int

    return dict(H_static=H_static, dH_mod=dH_f,
                nA=nA_f, nB=nB_f, nc=nc_f,
                H_A=H_A_f, H_B=H_B_f, H_c=H_c_f, H_int=H_int)

## 4. Identifying the dressed computational states

The qubit-coupler interaction hybridises the bare product states, so the
eigenstates of `H_static` are dressed states with some admixture of coupler
excitation.  The four computational states
$|gg,0\rangle$, $|gf,0\rangle$, $|fg,0\rangle$, $|ff,0\rangle$
are identified by finding, for each bare product state, the eigenstate of
`H_static` with the largest overlap.

In [ ]:
def bare_state(iA, iB, ic):
    |iA> x |iB> x |ic> as a full-system ket.
    return qt.tensor(qt.basis(nlev_q, iA),
                     qt.basis(nlev_q, iB),
                     qt.basis(nlev_c, ic))

def dressed_states(H_static, bare_list):
    For each bare ket in bare_list, return the eigenstate of H_static
    with the largest overlap.
    evals, evecs = H_static.eigenstates()
    out_states, out_energies, used = [], [], set()
    for b in bare_list:
        ov = np.array([abs((es.dag() * b)[0, 0])**2 for es in evecs])
        for k in used:
            ov[k] = -1
        idx = int(np.argmax(ov))
        used.add(idx)
        out_states.append(evecs[idx])
        out_energies.append(evals[idx])
    return out_states, np.array(out_energies)

def computational_subspace(H_static):
    bare = [bare_state(0, 0, 0),
            bare_state(0, 2, 0),
            bare_state(2, 0, 0),
            bare_state(2, 2, 0)]
    return dressed_states(H_static, bare)

## 5. Effective $J_{\rm mod}$ from $\Phi_{\rm ac}$

In the rotating frame at the drive frequency, the transition rate between the
dressed $|gf,0\rangle$ and $|fg,0\rangle$ states is set by the off-diagonal
matrix element of the modulation operator.  To leading order in $\Phi_{\rm ac}$,

$$J_{\rm mod}^{\rm eff}
   \;=\; \tfrac{1}{2}\,\Phi_{\rm ac}\,
   \bigl|\langle gf,0|_d\,(\partial H/\partial\Phi_c)\,|fg,0\rangle_d\bigr|,$$

where the factor of $\tfrac{1}{2}$ arises from the rotating-wave decomposition
of $\cos(\omega_d t)$.  This formula provides a direct mapping between
the experimental control knob $\Phi_{\rm ac}$ and the effective XY coupling
strength, and is used to cross-check the full time-domain simulation.

In [ ]:
def effective_Jmod(phi_ac, phi_dc):
    Returns the effective J_mod (GHz) driving |gf,0>_d <-> |fg,0>_d.
    sys = build_system(phi_dc)
    comp, _ = computational_subspace(sys["H_static"])
    gf, fg = comp[1], comp[2]
    matel = (gf.dag() * sys["dH_mod"] * fg)[0, 0]
    return 0.5 * phi_ac * abs(matel)

## 6. Gate simulation

The time-dependent Hamiltonian is

$$ H(t) \;=\; H_{\rm static}(\Phi_{\rm dc})
   \;+\; \Phi_{\rm ac}\,s(t)\,\cos(\omega_d t)\,
         \bigl[\partial H/\partial\Phi_c\bigr]_{\Phi_{\rm dc}}, $$

with a $\sin^2(\pi t / T_{\rm gate})$ pulse envelope $s(t)$.  The drive
frequency $\omega_d = \omega_{gf,A}^{(d)} - \omega_{gf,B}^{(d)}$ is read
from the dressed spectrum and therefore already incorporates the
coupler-induced Lamb shift.  An optional `detuning_GHz` offset is available
for AC-Stark compensation scans.

The full unitary propagator is obtained via `qt.propagator`, then
transformed to the interaction picture with respect to `H_static`.  The
result is projected onto the four dressed computational states, single-qubit
Z phases are analytically removed, and fidelity is computed against the
ideal iSWAP.

In [ ]:
IDEAL_ISWAP = qt.Qobj([[1, 0, 0, 0],
                       [0, 0, -1j, 0],
                       [0, -1j, 0, 0],
                       [0, 0, 0, 1]])

def simulate_gate(phi_ac, phi_dc, T_gate,
                  detuning_GHz=0.0, ramp_frac=0.0,
                  n_t=2001, verbose=False):
    Simulate the parametric iSWAP via explicit coupler flux modulation.

    Parameters
    ----------
    phi_ac : float
        Peak AC coupler-flux modulation amplitude, in units of Phi_0.
    phi_dc : float
        DC coupler-flux bias, in units of Phi_0.  Must satisfy 0 <= phi_dc < 0.5.
    T_gate : float
        Gate duration in ns.
    detuning_GHz : float
        Additive shift to the drive frequency (for AC-Stark scans).
    ramp_frac : float
        Fraction of T_gate spent ramping coupler flux DC bias in/out.  Set
        to 0 to assume the DC bias is already in place at t=0 (which is
        what the simple sin^2 envelope above models).
    n_t : int
        Number of time samples used by mesolve.

    Returns
    -------
    dict with keys: F, error, leakage_ee, leakage_coupler, J_mod_eff_MHz,
        U_int_4x4 (Z-corrected effective gate in comp. basis), times,
        populations (dict of np.arrays).
    
    sys = build_system(phi_dc)
    H_static = sys["H_static"]
    dH_mod   = sys["dH_mod"]

    bare5 = [bare_state(0, 0, 0), bare_state(0, 2, 0),
             bare_state(2, 0, 0), bare_state(2, 2, 0),
             bare_state(1, 1, 0)]
    dressed_all, energies_all = dressed_states(H_static, bare5)
    gg, gf, fg, ff, ee_dressed = dressed_all
    E_gg, E_gf, E_fg, E_ff, _  = energies_all

    omega_d = abs(E_gf - E_fg) + detuning_GHz

    J_mod_eff = 0.5 * phi_ac * abs((gf.dag() * dH_mod * fg)[0, 0])

    def envelope(t, args):
        T = args["T_gate"]
        if t < 0 or t > T:
            return 0.0
        return np.sin(np.pi * t / T)**2

    def drive_coeff(t, args):
        return args["phi_ac"] * envelope(t, args) * np.cos(2*np.pi*args["wd"]*t)

    H_td = [2*np.pi*H_static, [2*np.pi*dH_mod, drive_coeff]]
    args = dict(T_gate=T_gate, phi_ac=phi_ac, wd=omega_d)

    tlist = np.linspace(0, T_gate, n_t)

    if verbose:
        print(f"[sim] phi_ac={phi_ac:.4f} Phi0, phi_dc={phi_dc:.4f} Phi0, "
              f"T_gate={T_gate:.1f} ns, wd={omega_d*1e3:.2f} MHz, "
              f"J_mod_eff={J_mod_eff*1e3:.3f} MHz")
    res = qt.propagator(H_td, tlist, c_op_list=[], args=args,
                        options=qt.Options(nsteps=200000, atol=1e-9, rtol=1e-7))
    U_T = res[-1]

    U0_T = (-1j * 2*np.pi * H_static * T_gate).expm()
    U_int = U0_T.dag() * U_T

    comp = [gg, gf, fg, ff]
    M = np.zeros((4, 4), dtype=complex)
    for i in range(4):
        for j in range(4):
            M[i, j] = (comp[i].dag() * U_int * comp[j])[0, 0]
    U_comp = qt.Qobj(M)

    phases = np.angle(np.diag(M))
    p_gg, p_gf, p_fg, p_ff = phases
    zA = -(p_fg - p_gg)
    zB = -(p_gf - p_gg)
    Z = np.diag([np.exp(1j*(0)),
                 np.exp(1j*(zB)),
                 np.exp(1j*(zA)),
                 np.exp(1j*(zA + zB))])
    global_phase = np.exp(-1j*p_gg)
    U_z = qt.Qobj(Z) * U_comp * global_phase

    d = 4
    F = (abs((IDEAL_ISWAP.dag() * U_z).tr())**2 + d) / (d * (d + 1))
    F = float(np.real(F))

    populations = {name: np.zeros(n_t) for name in ("gg", "gf", "fg", "ff",
                                                    "ee", "coupler_excited")}
    coupler_excited_proj = qt.tensor(qt.qeye(nlev_q), qt.qeye(nlev_q),
                                     qt.qeye(nlev_c) - qt.basis(nlev_c, 0)*qt.basis(nlev_c, 0).dag())

    psi0 = gf
    out  = qt.mesolve(H_td, psi0, tlist, c_ops=[],
                       e_ops=[gg*gg.dag(), gf*gf.dag(), fg*fg.dag(), ff*ff.dag(),
                              ee_dressed*ee_dressed.dag(), coupler_excited_proj],
                       args=args,
                       options=qt.Options(nsteps=200000, atol=1e-9, rtol=1e-7))
    for k, name in enumerate(("gg", "gf", "fg", "ff", "ee", "coupler_excited")):
        populations[name] = np.real(out.expect[k])

    leakage_ee       = float(np.max(populations["ee"]))
    leakage_coupler  = float(np.max(populations["coupler_excited"]))

    return dict(
        F=F,
        error=1.0 - F,
        leakage_ee=leakage_ee,
        leakage_coupler=leakage_coupler,
        J_mod_eff_MHz=J_mod_eff * 1e3,
        U_z=U_z,
        omega_d_GHz=omega_d,
        times=tlist,
        populations=populations,
    )

## 7. Single operating point

A representative run at $\Phi_{\rm ac} = 10\,\mathrm{m}\Phi_0$,
$\Phi_{\rm dc} = 0.25\,\Phi_0$, $T_{\rm gate} = 400\,\mathrm{ns}$.
The printed $J_{\rm mod}^{\rm eff}$ gives the equivalent two-qubit
coupling strength for this flux operating point.

In [ ]:
t0 = time.time()
res = simulate_gate(phi_ac=0.010, phi_dc=0.25, T_gate=400.0, verbose=True)
print(f"\nelapsed: {time.time()-t0:.1f} s")

print(f"\nfidelity (Z-corrected):  F = {res['F']:.5f}")
print(f"error rate:              1-F = {res['error']:.2e}")
print(f"derived J_mod_eff:       {res['J_mod_eff_MHz']:.3f} MHz")
print(f"peak leakage to |ee>:    {res['leakage_ee']:.2e}")
print(f"peak coupler excitation: {res['leakage_coupler']:.2e}")

fig, ax = plt.subplots(figsize=(7, 4))
for name in ("gg", "gf", "fg", "ff", "ee", "coupler_excited"):
    ax.plot(res["times"], res["populations"][name], label=name)
ax.set_xlabel("t  (ns)")
ax.set_ylabel("population")
ax.set_title(r"Starting from $|gf,0\rangle_d$")
ax.set_ylim([-0.02, 1.02])
ax.legend(loc="center right", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Gate-time sweep at fixed flux

At a fixed operating point $(\Phi_{\rm ac}, \Phi_{\rm dc})$, the fidelity
oscillates with gate time as the gf-fg population undergoes Rabi-like
exchange at rate $J_{\rm mod}^{\rm eff}$.  The first maximum occurs near
$T_{\rm gate} \approx \pi / (2 J_{\rm mod}^{\rm eff})$, which is the
iSWAP condition.

In [ ]:
phi_ac_fix = 0.010
phi_dc_fix = 0.25
T_list = np.linspace(200, 800, 13)

F_list, err_list, leak_ee, leak_c = [], [], [], []
for T in T_list:
    r = simulate_gate(phi_ac_fix, phi_dc_fix, T_gate=T, n_t=1201)
    F_list.append(r["F"]); err_list.append(r["error"])
    leak_ee.append(r["leakage_ee"]); leak_c.append(r["leakage_coupler"])
    print(f"  T={T:6.1f} ns   F={r['F']:.4f}   leak_ee={r['leakage_ee']:.1e}   "
          f"leak_c={r['leakage_coupler']:.1e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(T_list, F_list, "o-")
ax1.set_xlabel(r"$T_{\rm gate}$ (ns)")
ax1.set_ylabel("Fidelity")
ax1.grid(alpha=0.3)
ax2.semilogy(T_list, err_list, "o-",  label=r"$1-F$")
ax2.semilogy(T_list, leak_ee, "s-",  label=r"max leak |ee>")
ax2.semilogy(T_list, leak_c,  "^-",  label="max coupler exc.")
ax2.set_xlabel(r"$T_{\rm gate}$ (ns)")
ax2.set_ylabel("error / leakage")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

## 9. Modulation amplitude sweep at fixed $(\Phi_{\rm dc}, T_{\rm gate})$

Increasing $\Phi_{\rm ac}$ raises $J_{\rm mod}^{\rm eff}$ linearly and
therefore shifts the fidelity maximum to shorter gate times, but it also
increases counter-rotating drive terms and transiently populates the
coupler.  This sweep identifies the amplitude that best matches a target
gate duration.

In [ ]:
phi_dc_fix = 0.25
T_gate_fix = 400.0
phi_ac_list = np.linspace(0.002, 0.030, 10)

F_list, err_list, J_list, leak_ee, leak_c = [], [], [], [], []
for pa in phi_ac_list:
    r = simulate_gate(pa, phi_dc_fix, T_gate=T_gate_fix, n_t=1201)
    F_list.append(r["F"]); err_list.append(r["error"])
    J_list.append(r["J_mod_eff_MHz"])
    leak_ee.append(r["leakage_ee"]); leak_c.append(r["leakage_coupler"])
    print(f"  phi_ac={pa*1e3:5.1f} mPhi0   J_mod_eff={r['J_mod_eff_MHz']:6.2f} MHz   "
          f"F={r['F']:.4f}   leak_c={r['leakage_coupler']:.1e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(np.array(phi_ac_list)*1e3, F_list, "o-")
ax1.set_xlabel(r"$\Phi_{ac}$  (m$\Phi_0$)")
ax1.set_ylabel("Fidelity")
ax1b = ax1.twinx()
ax1b.plot(np.array(phi_ac_list)*1e3, J_list, "s--", color="tab:red", alpha=0.5)
ax1b.set_ylabel(r"$J_{\rm mod}^{\rm eff}$ (MHz)", color="tab:red")
ax1.grid(alpha=0.3)
ax2.semilogy(np.array(phi_ac_list)*1e3, err_list, "o-", label=r"$1-F$")
ax2.semilogy(np.array(phi_ac_list)*1e3, leak_ee, "s-", label="leak |ee>")
ax2.semilogy(np.array(phi_ac_list)*1e3, leak_c,  "^-", label="leak coupler")
ax2.set_xlabel(r"$\Phi_{ac}$  (m$\Phi_0$)")
ax2.set_ylabel("error / leakage")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

## 10. 2D fidelity map: $\Phi_{\rm ac}$ vs $T_{\rm gate}$

A joint scan over modulation amplitude and gate duration at fixed
$\Phi_{\rm dc}$ reveals the expected iSWAP stripes: diagonal ridges
of high fidelity along the locus $T_{\rm gate} \cdot J_{\rm mod}^{\rm eff}(\Phi_{\rm ac}) = \pi/2$.
Larger $\Phi_{\rm ac}$ compresses the stripes toward shorter times and
gradually raises the leakage floor due to coupler dressing and
counter-rotating contributions.

In [ ]:
phi_dc_fix = 0.25
phi_ac_list = np.linspace(0.004, 0.020, 6)
T_list      = np.linspace(200, 700, 8)

F_grid = np.zeros((len(phi_ac_list), len(T_list)))
for i, pa in enumerate(phi_ac_list):
    for j, T in enumerate(T_list):
        r = simulate_gate(pa, phi_dc_fix, T_gate=T, n_t=801)
        F_grid[i, j] = r["F"]
    print(f" row {i+1}/{len(phi_ac_list)} done")

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(F_grid, origin="lower", aspect="auto",
               extent=[T_list[0], T_list[-1],
                       phi_ac_list[0]*1e3, phi_ac_list[-1]*1e3],
               cmap="viridis", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Fidelity")
ax.set_xlabel(r"$T_{\rm gate}$ (ns)")
ax.set_ylabel(r"$\Phi_{ac}$ (m$\Phi_0$)")
ax.set_title(rf"Fidelity, $\Phi_{{dc}} = {phi_dc_fix}\,\Phi_0$")
plt.tight_layout(); plt.show()